# 6. Save Population-Level Permutations

This notebook computes population-level and RDM-based drift measures from the simulated voxel responses generated in the previous notebook.

It also generates permutation-based null distributions by shuffling session order.

The outputs are later used for plotting, statistical testing, and comparing observed drift against shuffled-session baselines.

# Notebook Position in the Pipeline

Previous notebook:

`5_sim_pop_response_expand.ipynb`

This notebook:

`6_save_perm_pop_expand.ipynb`

Next notebook is for plotting

# Main Idea

The previous notebook generated predicted responses for the same image set across all session-specific models.

This notebook asks:

1. Are population response patterns stable across sessions?
2. Are image-level representational geometries stable across sessions?
3. Does similarity decrease as sessions become farther apart in time?
4. Is the observed pattern stronger than expected after shuffling session order?

The key null hypothesis is:

Session order does not matter.

# Inputs

This notebook uses two main input files.

## 1. Combined regression file

Generated by:

`3_Combine_Session__ROIs.ipynb`

File format:

`regressSessCombineROI_sub{subject}{normalization}.pkl`

Used for:

- pRF metadata
- ROI information
- voxel filtering based on pRF R²

## 2. Simulated population responses

Generated by:

`5_sim_pop_response_expand.ipynb`

File format:

`simPopResp_v{visualRegion}_sub{subject}{normalization}.npz`

Used for:

- predicted voxel responses from the non-orientation model
- predicted voxel responses from the orientation model
- image IDs used for the simulation

# What This Notebook Computes

For each subject and visual region, this notebook computes:

## Population response similarity

For each image, it compares the voxel population response pattern across sessions.

This produces a session × session similarity matrix.

## RDM construction

For each session, it builds an image-by-image representational dissimilarity matrix (RDM) from the simulated population responses.

The RDM can be built using:

- correlation distance
- Euclidean distance

## RDM comparison across sessions

The session-specific RDMs are compared using:

- Spearman correlation
- RMSE

## Session-lag curves

The session × session matrices are collapsed by temporal distance between sessions.

For example:

- lag 1: session pairs separated by one session
- lag 2: session pairs separated by two sessions
- lag 3: session pairs separated by three sessions

## Permutations

The same lag curves are recomputed after randomly shuffling session order.

This creates a null distribution for testing whether the real chronological order shows stronger drift than expected by chance.

# Outputs

This notebook saves one compressed `.npz` file per:

- subject
- visual region
- normalization type
- RDM construction metric
- RDM comparison metric

Output filename format:

`permPop_N{nperms}_v{visualRegion}_sub{subject}{normalization}_{metricTag}.npz`

Example:

`permPop_N1000_v1_sub1_equalStd_rdmCorrDist_cmpSpearman.npz`

Each file contains:

- observed population similarity matrices
- observed RDM similarity / distance matrices
- lag-averaged observed curves
- permutation orders
- permutation-based null distributions
- metadata such as visual region, number of good voxels, and metric type

# Run Code

In [ ]:
# save_perm_pop_expand_all_rdm_combos.py

import os
import numpy as np
import pickle
from scipy.linalg import toeplitz
from scipy.spatial.distance import pdist
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# =========================
# CONFIGURATION
# ============================================================
# These settings control which data are analyzed and how many
# shuffled-session null samples are generated.

# DEFAULT_TO_ZSCORE:
#     normalization suffix used to load simulated responses.

# NPERMS:
#     number of random session-order permutations.

# R2THRESH:
#     optional pRF R² threshold for selecting reliable voxels.

# VREGS:
#     visual regions to analyze.

# RDM_COMBOS:
#     combinations of RDM construction metric and RDM comparison metric.
# =========================
BASE_FOLDER = "/content/drive/My Drive/V1_Drift/repDrift_expand"

DEFAULT_TO_ZSCORE = 3
NPERMS = 1000
R2THRESH = 0.0
FIXED_FIRST = False
OVERWRITE = True

VREGS = [1, 2, 3, 4]

# RDM construction × RDM comparison
RDM_COMBOS = [
    ("correlation", "spearman"),  # article-like: RDM = 1 - Pearson, compare with Spearman
    ("correlation", "rmse"),      # RDM = 1 - Pearson, compare with RMSE
    ("euclidean", "spearman"),    # RDM = Euclidean distance, compare with Spearman
    ("euclidean", "rmse"),        # RDM = Euclidean distance, compare with RMSE
]


# =========================
# PATH HELPERS
# =========================
def _zstr(to_zscore: int) -> str:
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]


def _comb_path(base_folder, isub, to_zscore):
    return os.path.join(
        base_folder,
        f"regressSessCombineROI_sub{isub}{_zstr(to_zscore)}.pkl"
    )


def _sim_path(base_folder, vreg, isub, to_zscore):
    return os.path.join(
        base_folder,
        f"simPopResp_v{vreg}_sub{isub}{_zstr(to_zscore)}.npz"
    )


def _metric_tag(rdm_build_metric, rdm_compare_metric):
    build_tag = {
        "correlation": "rdmCorrDist",
        "euclidean": "rdmEuclidean"
    }[rdm_build_metric]

    compare_tag = {
        "spearman": "cmpSpearman",
        "rmse": "cmpRMSE"
    }[rdm_compare_metric]

    return f"{build_tag}_{compare_tag}"


def _out_path(base_folder, vreg, isub, to_zscore, nperms,
              r2thresh, fixedFirst, rdm_build_metric, rdm_compare_metric):

    fixed = "_fixedFirst_" if fixedFirst else "_"

    r2thresh_str = ""
    if r2thresh > 0:
        r2thresh_str = f"_r2thresh{r2thresh:g}"

    metric_tag = _metric_tag(rdm_build_metric, rdm_compare_metric)

    return os.path.join(
        base_folder,
        f"permPop{fixed}N{nperms}_v{vreg}_sub{isub}"
        f"{_zstr(to_zscore)}{r2thresh_str}_{metric_tag}.npz"
    )


def _discover_subjects_with_inputs(base_folder, vreg, to_zscore):
    subjects = []
    for isub in range(1, 9):
        if os.path.exists(_comb_path(base_folder, isub, to_zscore)) and \
           os.path.exists(_sim_path(base_folder, vreg, isub, to_zscore)):
            subjects.append(isub)
    return subjects


# =========================
# METRIC HELPERS
# =========================
def _nanmean(x, axis=None):
    return np.nanmean(x, axis=axis)


def _pearson_masked(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan

    x = x[mask]
    y = y[mask]

    x = x - x.mean()
    y = y - y.mean()

    denom = np.linalg.norm(x) * np.linalg.norm(y)
    if denom == 0:
        return np.nan

    return float((x @ y) / denom)


def _spearman_masked(x, y):
    from scipy.stats import rankdata

    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan

    xr = rankdata(x[mask], method="average")
    yr = rankdata(y[mask], method="average")

    return _pearson_masked(xr, yr)


def _rmse_masked(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() == 0:
        return np.nan

    diff = x[mask] - y[mask]
    return float(np.sqrt(np.mean(diff ** 2)))


# ============================================================
# COMPARE SESSION-SPECIFIC RDMS
# ============================================================
# Each session has one vectorized RDM.

# This function compares every pair of session RDMs and returns
# a session × session matrix.

# Spearman:
#     high value = similar representational geometry

# RMSE:
#     high value = more different representational geometry
# ============================================================
def compare_session_rdms(sessVec, compare_metric):
    """
    sessVec: [nsessions, n_rdm_entries]
    Each row is one vectorized RDM.

    compare_metric:
      spearman = rank-based RDM similarity
      rmse     = absolute RDM distance

    For Spearman:
      higher = more similar

    For RMSE:
      higher = more different / more drift
    """
    ns = sessVec.shape[0]
    R = np.full((ns, ns), np.nan, dtype=float)

    for i in range(ns):
        if compare_metric == "spearman":
            R[i, i] = 1.0
        elif compare_metric == "rmse":
            R[i, i] = 0.0
        else:
            raise ValueError(f"Unknown compare_metric: {compare_metric}")

        xi = np.asarray(sessVec[i], dtype=float)

        for j in range(i + 1, ns):
            xj = np.asarray(sessVec[j], dtype=float)

            if compare_metric == "spearman":
                val = _spearman_masked(xi, xj)
            elif compare_metric == "rmse":
                val = _rmse_masked(xi, xj)

            R[i, j] = val
            R[j, i] = val

    return R


# ============================================================
# MAIN FUNCTION: SAVE POPULATION-LEVEL PERMUTATIONS
# ============================================================
# For one subject and one visual region, this function:
#
#     1. Loads simulated voxel responses.
#     2. Optionally filters voxels by pRF R².
#     3. Computes population response similarity across sessions.
#     4. Builds an RDM for each session.
#     5. Compares RDMs across sessions.
#     6. Collapses session-by-session matrices by temporal lag.
#     7. Shuffles session order to create permutation null distributions.
#     8. Saves all observed and permuted results.
# =========================
def save_perm_pop_expand_npz(
    isub,
    vreg,
    rdm_build_metric,
    rdm_compare_metric,
    to_zscore=DEFAULT_TO_ZSCORE,
    nperms=NPERMS,
    r2thresh=R2THRESH,
    fixedFirst=FIXED_FIRST,
    base_folder=BASE_FOLDER
):
    print(
        f"\nRunning subject {isub}, vreg {vreg}, "
        f"RDM build = {rdm_build_metric}, compare = {rdm_compare_metric}"
    )

    # ---------- Load combined regression data ----------
    comb_path = _comb_path(base_folder, isub, to_zscore)

    with open(comb_path, "rb") as f:
        C = pickle.load(f)

    nsplits = int(C["nsplits"])
    ns_no_mean = nsplits - 1
    allRoiPrf = C["allRoiPrf"]

    # ---------- Load simulated population responses ----------
    sim_path = _sim_path(base_folder, vreg, isub, to_zscore)
    S = np.load(sim_path, allow_pickle=True)

    voxSessResp = S["voxSessResp"]          # [vox, session, image]
    voxSessRespOri = S["voxSessRespOri"]    # [vox, session, image]
    simImgs = S["simImgs"]

    nimg = int(simImgs.shape[0])

    # Use first true sessions only, excluding mean session if present
    Xfull = voxSessResp[:, :ns_no_mean, :]
    Xofull = voxSessRespOri[:, :ns_no_mean, :]

    ns = Xfull.shape[1]
    minSessions = ns

    # ---------- Voxel filtering ----------
    roi_prf = allRoiPrf[vreg - 1]

    if "r2" in roi_prf:
        r2vec = np.asarray(roi_prf["r2"]).reshape(-1)
        good = np.isfinite(r2vec) & (r2vec > r2thresh)
    else:
        if r2thresh > 0:
            raise KeyError("r2 missing, cannot apply r2 threshold.")
        good = np.ones(Xfull.shape[0], dtype=bool)

    nvox_good = int(np.sum(good))
    if nvox_good == 0:
        raise RuntimeError(f"No voxels pass r2thresh={r2thresh}")

    X = Xfull[good, :, :].astype(float)
    Xo = Xofull[good, :, :].astype(float)

    # =========================
    # 1. POPULATION RESPONSE SIMILARITY
    # =========================
    imgCorr = np.full((nimg, ns, ns), np.nan)
    imgCorrOri = np.full((nimg, ns, ns), np.nan)

    for ii in range(nimg):
        M = X[:, :, ii].T
        Mo = Xo[:, :, ii].T

        imgCorr[ii] = np.corrcoef(M, rowvar=True)
        imgCorrOri[ii] = np.corrcoef(Mo, rowvar=True)

    avgImgCorrMat = _nanmean(imgCorr, axis=0)
    avgImgCorrMatOri = _nanmean(imgCorrOri, axis=0)

    # =========================
    # 2. RDM CONSTRUCTION
    # =========================
    nvec = nimg * (nimg - 1) // 2

    sessVec = np.full((ns, nvec), np.nan)
    sessVecOri = np.full((ns, nvec), np.nan)

    for ss in range(ns):
        Ms = X[:, ss, :].T
        Mso = Xo[:, ss, :].T

        # If rdm_build_metric == "correlation":
        # scipy pdist returns correlation distance = 1 - Pearson r.
        #
        # If rdm_build_metric == "euclidean":
        # scipy pdist returns Euclidean distances between image population vectors.
        sessVec[ss] = pdist(Ms, metric=rdm_build_metric)
        sessVecOri[ss] = pdist(Mso, metric=rdm_build_metric)

    # =========================
    # 3. RDM COMPARISON ACROSS SESSIONS
    # =========================
    R = compare_session_rdms(sessVec, compare_metric=rdm_compare_metric)
    Ro = compare_session_rdms(sessVecOri, compare_metric=rdm_compare_metric)

    # =========================
    # 4. COLLAPSE BY SESSION LAG
    # =========================
    distMatrix = toeplitz(np.arange(minSessions))

    betweenSessImg = np.full(minSessions - 1, np.nan)
    betweenSessImgOri = np.full(minSessions - 1, np.nan)

    betweenSessDist = np.full(minSessions - 1, np.nan)
    betweenSessDistOri = np.full(minSessions - 1, np.nan)

    for d in range(1, minSessions):
        mask = distMatrix == d

        # Population response similarity
        betweenSessImg[d - 1] = _nanmean(avgImgCorrMat[mask])
        betweenSessImgOri[d - 1] = _nanmean(avgImgCorrMatOri[mask])

        # RDM comparison metric
        betweenSessDist[d - 1] = _nanmean(R[mask])
        betweenSessDistOri[d - 1] = _nanmean(Ro[mask])

    # =========================
    # 5. SESSION PERMUTATIONS
    # =========================
    rng = np.random.default_rng(1)

    if fixedFirst:
        permOrders = np.array([
            np.concatenate(([0], rng.permutation(minSessions - 1) + 1))
            for _ in range(nperms)
        ], dtype=int)
    else:
        permOrders = np.array([
            rng.permutation(minSessions)
            for _ in range(nperms)
        ], dtype=int)

    # Population permutations
    imgCorrMatPerm = np.full((nperms, minSessions, minSessions), np.nan)
    imgCorrMatOriPerm = np.full((nperms, minSessions, minSessions), np.nan)
    betweenSessImgPerm = np.full((nperms, minSessions - 1), np.nan)
    betweenSessImgOriPerm = np.full((nperms, minSessions - 1), np.nan)

    # RDM permutations
    betweenSessCorrPerm = np.full((nperms, minSessions, minSessions), np.nan)
    betweenSessCorrOriPerm = np.full((nperms, minSessions, minSessions), np.nan)
    betweenSessDistPerm = np.full((nperms, minSessions - 1), np.nan)
    betweenSessDistOriPerm = np.full((nperms, minSessions - 1), np.nan)

    for ip, ordv in enumerate(permOrders):
        # Population response matrix permutation
        tmp_img = avgImgCorrMat[np.ix_(ordv, ordv)]
        tmp_imgo = avgImgCorrMatOri[np.ix_(ordv, ordv)]

        imgCorrMatPerm[ip] = tmp_img
        imgCorrMatOriPerm[ip] = tmp_imgo

        # RDM matrix permutation
        tmp_rdm = R[np.ix_(ordv, ordv)]
        tmp_rdmo = Ro[np.ix_(ordv, ordv)]

        betweenSessCorrPerm[ip] = tmp_rdm
        betweenSessCorrOriPerm[ip] = tmp_rdmo

        for d in range(1, minSessions):
            mask = distMatrix == d

            betweenSessImgPerm[ip, d - 1] = _nanmean(tmp_img[mask])
            betweenSessImgOriPerm[ip, d - 1] = _nanmean(tmp_imgo[mask])

            betweenSessDistPerm[ip, d - 1] = _nanmean(tmp_rdm[mask])
            betweenSessDistOriPerm[ip, d - 1] = _nanmean(tmp_rdmo[mask])

    # =========================
    # SAVE
    # =========================
    out = dict(
        # meta
        toZscore=int(to_zscore),
        r2thresh=float(r2thresh),
        subjects=np.array([isub], dtype=int),
        vreg=int(vreg),

        rdmBuildMetric=rdm_build_metric,
        rdmCompareMetric=rdm_compare_metric,
        metricTag=_metric_tag(rdm_build_metric, rdm_compare_metric),

        minSessions=int(minSessions),
        distMatrix=distMatrix.astype(int),
        subSessions=np.ones(minSessions, dtype=int),
        numGoodVox=np.array(nvox_good, dtype=int),

        # population response similarity
        avgImgCorrMat=avgImgCorrMat,
        avgImgCorrMatOri=avgImgCorrMatOri,
        betweenSessImg=betweenSessImg,
        betweenSessImgOri=betweenSessImgOri,

        # RDM across-session metric
        betweenSessCorr=R,
        betweenSessCorrOri=Ro,
        betweenSessDist=betweenSessDist,
        betweenSessDistOri=betweenSessDistOri,

        # permutations
        permOrders=permOrders,

        imgCorrMatPerm=imgCorrMatPerm,
        imgCorrMatOriPerm=imgCorrMatOriPerm,
        betweenSessImgPerm=betweenSessImgPerm,
        betweenSessImgOriPerm=betweenSessImgOriPerm,

        betweenSessCorrPerm=betweenSessCorrPerm,
        betweenSessCorrOriPerm=betweenSessCorrOriPerm,
        betweenSessDistPerm=betweenSessDistPerm,
        betweenSessDistOriPerm=betweenSessDistOriPerm,
    )

    out_path = _out_path(
        base_folder, vreg, isub, to_zscore,
        nperms, r2thresh, fixedFirst,
        rdm_build_metric, rdm_compare_metric
    )

    np.savez_compressed(out_path, **out)
    print(f"Saved: {out_path}")

    return out_path


# =========================
# BATCH RUNNERS
# ============================================================
# Run the population permutation analysis across:
#
#     - all available subjects
#     - all requested visual regions
#     - all RDM metric combinations
#
# Subjects are discovered automatically only if both required
# input files exist.
# ============================================================
def run_perm_pop_all_subjects_one_combo(
    vreg,
    rdm_build_metric,
    rdm_compare_metric,
    to_zscore=DEFAULT_TO_ZSCORE,
    nperms=NPERMS,
    r2thresh=R2THRESH,
    fixedFirst=FIXED_FIRST,
    overwrite=OVERWRITE,
    base_folder=BASE_FOLDER
):
    subjects = _discover_subjects_with_inputs(base_folder, vreg, to_zscore)

    print(
        f"\nVREG {vreg} | "
        f"RDM build {rdm_build_metric} | compare {rdm_compare_metric}"
    )
    print(f"Subjects with inputs: {subjects}")

    out_paths = []

    for isub in subjects:
        out_path = _out_path(
            base_folder, vreg, isub, to_zscore,
            nperms, r2thresh, fixedFirst,
            rdm_build_metric, rdm_compare_metric
        )

        if os.path.exists(out_path) and not overwrite:
            print(f"Subject {isub}: exists, skip")
            continue

        try:
            p = save_perm_pop_expand_npz(
                isub=isub,
                vreg=vreg,
                rdm_build_metric=rdm_build_metric,
                rdm_compare_metric=rdm_compare_metric,
                to_zscore=to_zscore,
                nperms=nperms,
                r2thresh=r2thresh,
                fixedFirst=fixedFirst,
                base_folder=base_folder
            )
            out_paths.append(p)

        except Exception as e:
            print(
                f"[ERROR] subject {isub}, vreg {vreg}, "
                f"build {rdm_build_metric}, compare {rdm_compare_metric}: {e}"
            )

    return out_paths


def run_all_rdm_combos_all_regions():
    all_outputs = []

    for rdm_build_metric, rdm_compare_metric in RDM_COMBOS:
        for vreg in VREGS:
            outs = run_perm_pop_all_subjects_one_combo(
                vreg=vreg,
                rdm_build_metric=rdm_build_metric,
                rdm_compare_metric=rdm_compare_metric,
                to_zscore=DEFAULT_TO_ZSCORE,
                nperms=NPERMS,
                r2thresh=R2THRESH,
                fixedFirst=FIXED_FIRST,
                overwrite=OVERWRITE,
                base_folder=BASE_FOLDER
            )
            all_outputs.extend(outs)

    print("\nDONE")
    print(f"Created/updated {len(all_outputs)} files.")
    return all_outputs


# =========================
# RUN
# =========================
if __name__ == "__main__":
    run_all_rdm_combos_all_regions()


# CHECK SAVED POPULATION PERMUTATION FILE
# ============================================================
 This cell verifies that the output generated by:
#
#     6_save_perm_pop_expand.ipynb
#
was saved correctly.
#
# Main goals:
#
     1. Confirm that the file exists.
     2. Verify that all expected variables were saved.
     3. Check array dimensions.
     4. Detect NaNs or corrupted outputs.
     5. Inspect observed population/RDM measures.
     6. Inspect permutation-based null distributions.
#
 This should be run after the permutation analysis finishes.


In [ ]:
import os
import numpy as np
from google.colab import drive

BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
path = os.path.join(BASE_FOLDER, "permPop_N1000_v1_sub1_spearman.npz")  # change name if needed

d = np.load(path, allow_pickle=True)

print("FILE:", path)
print("\nKeys:")
for k in d.files:
    print(" -", k)

print("\nBasic shapes:")
for k in d.files:
    arr = d[k]
    if isinstance(arr, np.ndarray):
        print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")
    else:
        print(f"{k}: type={type(arr)}")



def show_stats(name):
    x = d[name]
    print(f"\n{name}")
    print(" shape:", x.shape)
    if np.issubdtype(x.dtype, np.number):
        finite = np.isfinite(x)
        print(" finite fraction:", finite.mean())
        if finite.any():
            xf = x[finite]
            print(" mean:", float(xf.mean()))
            print(" std :", float(xf.std()))
            print(" min :", float(xf.min()))
            print(" max :", float(xf.max()))

for k in [
    "betweenSessImg", "betweenSessImgOri",
    "betweenSessDist", "betweenSessDistOri",
    "avgImgCorrMat", "avgImgCorrMatOri",
    "betweenSessCorr", "betweenSessCorrOri",
    "betweenSessImgPerm", "betweenSessDistPerm",
    "permOrders", "numGoodVox", "minSessions"
]:
    if k in d.files:
        show_stats(k)



print("\nminSessions:", int(d["minSessions"]))
print("numGoodVox :", int(d["numGoodVox"]))

print("\nbetweenSessImg:")
print(d["betweenSessImg"])

print("\nbetweenSessDist:")
print(d["betweenSessDist"])

print("\nFirst 5 permuted betweenSessDist rows:")
print(d["betweenSessDistPerm"][:5, :5])

print("\nTop-left of avgImgCorrMat:")
print(d["avgImgCorrMat"][:5, :5])

print("\nTop-left of betweenSessCorr:")
print(d["betweenSessCorr"][:5, :5])